# Export Qwen3.5-4B to ExecuTorch .pte (Unsloth Style)

This notebook follows the same flow as your Qwen3-4B Unsloth notebook:
1. Load with Unsloth (attempt QAT)
2. Save TorchAO checkpoint
3. Convert weights for ExecuTorch
4. Export to `.pte`

Notes:
- This uses your `qwen3_5_phase2` ExecuTorch branch.
- Qwen3.5 export is currently fp32/static-shape in this path.
- 4B fp32 `.pte` can be very large (15GB+).


## Step 1: Install Dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install torchao==0.14.0 safetensors ruamel.yaml tabulate pandas


In [ ]:
%%bash
set -e
cd /content
if [ ! -d executorch ]; then
  git clone https://github.com/Phineas1500/executorch.git
fi
cd /content/executorch
git fetch --all
git checkout qwen3_5_phase2
git pull --ff-only origin qwen3_5_phase2
git submodule sync --recursive
git submodule update --init --recursive
./install_executorch.sh --editable


In [ ]:
%%bash
set -e
cd /content/executorch
cmake -S third-party/flatbuffers -B /tmp/flatbuffers-build -DFLATBUFFERS_BUILD_FLATC=ON -DFLATBUFFERS_BUILD_TESTS=OFF -DFLATBUFFERS_BUILD_FLATHASH=OFF
cmake --build /tmp/flatbuffers-build -j


## Step 2: Load Qwen3.5-4B with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

candidates = [
    'unsloth/Qwen3.5-4B',
    'Qwen/Qwen3.5-4B',
]

last_err = None
model = tokenizer = None
for model_name in candidates:
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=1024,
            full_finetuning=True,
            qat_scheme='phone-deployment',
        )
        print(f'Loaded with QAT: {model_name}')
        break
    except Exception as e:
        last_err = e

if model is None:
    print('QAT load failed, retrying without qat_scheme...')
    for model_name in candidates:
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=model_name,
                max_seq_length=1024,
                full_finetuning=True,
            )
            print(f'Loaded without QAT: {model_name}')
            break
        except Exception as e:
            last_err = e

if model is None:
    raise RuntimeError(f'Failed to load Qwen3.5-4B with Unsloth: {last_err}')


## Step 3: Save in TorchAO Format

In [ ]:
print('Saving model in TorchAO format...')
model.save_pretrained_torchao('qwen3_5_4b_model', tokenizer=tokenizer)
print('Done!')


## Step 4: Convert Weights for ExecuTorch

In [ ]:
%%bash
set -e
cd /content/executorch
python -m executorch.examples.models.qwen3_5.convert_weights /content/qwen3_5_4b_model /content/qwen3_5_4b_weights.pth
ls -lh /content/qwen3_5_4b_weights.pth


## Step 5: Export to .pte (Smoke, No Backend)

In [ ]:
%%bash
set -e
cd /content/executorch
OMP_NUM_THREADS=1 TORCHINDUCTOR_COMPILE_THREADS=1 \
PATH=/tmp/flatbuffers-build:$PATH \
python -m extension.llm.export.export_llm \
  --config examples/models/qwen3_5/config/qwen3_5_xnnpack_fp32.yaml \
  +base.model_class=qwen3_5_4b \
  +base.params=examples/models/qwen3_5/config/4b_config.json \
  +base.checkpoint=/content/qwen3_5_4b_weights.pth \
  backend.xnnpack.enabled=False \
  export.max_seq_length=1 \
  export.max_context_length=1 \
  +export.output_name=/content/qwen3_5_4b_no_backend_smoke.pte


## Step 6: Export to .pte (XNNPACK fp32)

In [ ]:
%%bash
set -e
cd /content/executorch
OMP_NUM_THREADS=1 TORCHINDUCTOR_COMPILE_THREADS=1 \
PATH=/tmp/flatbuffers-build:$PATH \
python -m extension.llm.export.export_llm \
  --config examples/models/qwen3_5/config/qwen3_5_xnnpack_fp32.yaml \
  +base.model_class=qwen3_5_4b \
  +base.params=examples/models/qwen3_5/config/4b_config.json \
  +base.checkpoint=/content/qwen3_5_4b_weights.pth \
  export.max_seq_length=128 \
  export.max_context_length=128 \
  +export.output_name=/content/qwen3_5_4b_xnnpack_fp32_128.pte


## Step 7: Check Outputs

In [ ]:
%%bash
set -e
ls -lh /content/qwen3_5_4b_*.pte
ls -lh /content/qwen3_5_4b_model/tokenizer.json || true
ls -lh /content/qwen3_5_4b_model/tokenizer_config.json || true


## Step 8: Download (Chunked for Large Files)

In [ ]:
%%bash
set -e
split -b 1024m /content/qwen3_5_4b_xnnpack_fp32_128.pte /content/qwen3_5_4b_xnnpack_fp32_128.pte.part.
ls -lh /content/qwen3_5_4b_xnnpack_fp32_128.pte.part.*


In [ ]:
import glob
from google.colab import files
for p in sorted(glob.glob('/content/qwen3_5_4b_xnnpack_fp32_128.pte.part.*')):
    files.download(p)

for p in ['/content/qwen3_5_4b_model/tokenizer.json', '/content/qwen3_5_4b_model/tokenizer_config.json']:
    try:
        files.download(p)
    except Exception as e:
        print(f'Skipped {p}: {e}')
